# CEG5305 Part 2, Lecture 1: the ablation homework, in Colab

This notebook runs [`ablation_base.py`](https://costa-nus.github.io/CEG5305_GenAI/part2/lecture-1/code/ablation_base.py), the small GPT from the lecture: 4 blocks, 4 heads, width 128, 818,241 parameters, trained on TinyShakespeare for 3,000 steps with three seeds.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. One run takes about 35 s on a T4, so one variant (three seeds) takes about 2 minutes. The CPU runtime also works, only slower.

In [ ]:
# 1. Get the base code
!wget -q -O ablation_base.py https://costa-nus.github.io/CEG5305_GenAI/part2/lecture-1/code/ablation_base.py
!python ablation_base.py --help | head -20

In [ ]:
# 2. The baseline, three seeds. Seed 1337 should give a validation loss of 1.765.
!python ablation_base.py --variant baseline --out baseline.json

## 3. Your change

Open `ablation_base.py` in the file panel on the left (double-click to edit). Add **one** entry to `VARIANTS`, either a new switch that you use in the model code, like the four class variants, or one value from `FIXED` that you change on purpose, for example `"deeper": {"n_layer": 12}`. Keep everything else as it is. Save, then run your variant.

In [ ]:
!python ablation_base.py --variant deeper --out mine.json   # replace "deeper" with your variant's name

In [ ]:
# 4. Compare: mean and range over the three seeds, and the training curves
import json
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for path in ["baseline.json", "mine.json"]:
    d = json.load(open(path))
    for name, r in d["results"].items():
        print(f"{name:12s} mean {r['mean']:.3f}  range {r['min']:.3f}-{r['max']:.3f}  ({r['parameters']:,} parameters)")
        runs = list(d["curves"][name].values())
        steps = runs[0]["step"]
        mean = [sum(c["train_loss"][i] for c in runs) / len(runs) for i in range(len(steps))]
        ax.plot(steps, mean, label=f"{name} (mean of {len(runs)} seeds)")
ax.set_xlabel("training step"); ax.set_ylabel("training loss (nats per character)")
ax.set_ylim(1, 4.5); ax.legend(); ax.grid(alpha=.3)
plt.show()